In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the new rate. The blue dots are the original samples at 8 Hz, and
the red crosses are the resampled points. Each cross sits on the dashed
line joining its two blue neighbors, which is exactly what linear
interpolation reads. The gray curve is the true signal.

In [ ]:
# hide
# autorun
FS1 = 8.0                           # the original rate, as in the figure above
DUR = 2.0                           # two full cycles of a 1 Hz sine
N = int(DUR * FS1)
X = np.sin(2 * np.pi * np.arange(N) / FS1)
T_SMOOTH = np.linspace(0.0, DUR, 600)
FS2_0 = 12.0                        # starting parameter

def resample(fs2, N=N, X=X, FS1=FS1):
    # the chapter's linear interpolation; the two cycles repeat, so the
    # neighbor after the last sample is the first one again
    M = int(round(N * fs2 / FS1))
    p = np.arange(M) * FS1 / fs2
    i = np.floor(p).astype(int)
    a = p - i
    y = (1 - a) * X[i] + a * X[(i + 1) % N]
    return np.arange(M) / fs2, y

def figure():
    fig = go.Figure()
    tm, ym = resample(FS2_0)
    fig.add_scatter(x=T_SMOOTH, y=np.sin(2 * np.pi * T_SMOOTH), mode="lines",
                    line=dict(color=STEEL, width=1.8))
    fig.add_scatter(x=np.arange(N + 1) / FS1, y=np.append(X, X[0]), mode="lines",
                    line=dict(color=BLUE, width=1.2, dash="dash"))
    fig.add_scatter(x=np.arange(N) / FS1, y=X, mode="markers",
                    marker=dict(color=BLUE, size=9))
    fig.add_scatter(x=tm, y=ym, mode="markers",
                    marker=dict(color=RED, size=10, symbol="x-thin",
                                line=dict(color=RED, width=2.4)))
    fig.update_xaxes(range=[-0.05, DUR + 0.05], title_text="Time (s)", fixedrange=True)
    fig.update_yaxes(range=[-1.2, 1.2], title_text="Amplitude", fixedrange=True)
    return fig

def controls(fig):
    fs2 = widgets.FloatSlider(description="New rate (Hz)", min=3, max=24,
                              value=FS2_0, step=1, readout_format=".0f")
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(fs2, resample=resample, N=N, FS1=FS1, readout=readout):
        tm, ym = resample(fs2)
        with fig.batch_update():
            fig.data[3].x, fig.data[3].y = tm, ym
        readout.value = (f"<span style='font-size:0.9em'>M = N &middot; "
                         f"f<sub>s</sub><sup>2</sup> / f<sub>s</sub><sup>1</sup> = "
                         f"{N} &middot; {fs2:.0f} / {FS1:.0f} = {len(tm)} samples"
                         f"</span>")

    widgets.interactive_output(update, {"fs2": fs2})
    return widgets.VBox([fs2, readout])

icm_plotly.show(figure, controls)